# UCI Adult Income Classification with a PyTorch MLP

This notebook trains a simple multilayer perceptron to predict whether a person's income is `>50K` or `<=50K` using the UCI Adult dataset.

## 1. Imports and configuration

In [ ]:
from collections.abc import Sequence
from pathlib import Path

import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
DATA_DIR = Path("data/adult")
TRAIN_PATH = DATA_DIR / "adult.data"
TEST_PATH = DATA_DIR / "adult.test"

BATCH_SIZE = 256
EPOCHS = 20
LEARNING_RATE = 1e-3
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 2. Load the Adult dataset

The target is `income`. The original test file has a metadata line at the top and labels ending with a period, so the loader handles both details.

In [ ]:
columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income",
]


def load_adult_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        names=columns,
        skipinitialspace=True,
        na_values="?",
        comment="|",
    )
    df["income"] = df["income"].str.replace(".", "", regex=False)
    return df.dropna().reset_index(drop=True)


train_df = load_adult_csv(TRAIN_PATH)
test_df = load_adult_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()

## 3. Preprocess features

Numerical columns are standardized using train-set statistics. Categorical columns are one-hot encoded after joining train and test features so both splits get the same columns.

In [ ]:
target_col = "income"
numeric_cols = [
    "age",
    "fnlwgt",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
]

x_train_raw = train_df.drop(columns=target_col)
x_test_raw = test_df.drop(columns=target_col)
y_train = (train_df[target_col] == ">50K").astype("float32")
y_test = (test_df[target_col] == ">50K").astype("float32")

combined = pd.concat([x_train_raw, x_test_raw], axis=0)
combined = pd.get_dummies(combined, columns=[c for c in combined.columns if c not in numeric_cols], dtype="float32")

x_train = combined.iloc[: len(train_df)].copy()
x_test = combined.iloc[len(train_df) :].copy()

means = x_train[numeric_cols].mean()
stds = x_train[numeric_cols].std().replace(0, 1)
x_train[numeric_cols] = (x_train[numeric_cols] - means) / stds
x_test[numeric_cols] = (x_test[numeric_cols] - means) / stds

x_train_tensor = torch.tensor(x_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values.reshape(-1, 1), dtype=torch.float32)
x_test_tensor = torch.tensor(x_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values.reshape(-1, 1), dtype=torch.float32)

print("Number of input features after preprocessing:", x_train_tensor.shape[1])
print("Positive class rate in train:", y_train_tensor.mean().item())
print("Positive class rate in test:", y_test_tensor.mean().item())

## 4. Build dataloaders

In [ ]:
train_loader = DataLoader(
    TensorDataset(x_train_tensor, y_train_tensor),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

test_loader = DataLoader(
    TensorDataset(x_test_tensor, y_test_tensor),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

## 5. Define the MLP

This is a binary classifier. It outputs one logit per row, then `BCEWithLogitsLoss` applies the sigmoid internally during training.

In [ ]:
class MLP(nn.Module):
    """A simple fully connected neural network for tabular/vector inputs."""

    def __init__(
        self,
        input_dim: int,
        hidden_dims: Sequence[int],
        output_dim: int,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()

        layers: list[nn.Module] = []
        previous_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(previous_dim, hidden_dim))
            layers.append(nn.ReLU())
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            previous_dim = hidden_dim

        layers.append(nn.Linear(previous_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


model = MLP(
    input_dim=x_train_tensor.shape[1],
    hidden_dims=(128, 64, 32),
    output_dim=1,
    dropout=0.2,
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

model

## 6. Train and evaluate

In [ ]:
def evaluate(model: nn.Module, loader: DataLoader) -> dict[str, float]:
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    true_positive = 0
    false_positive = 0
    false_negative = 0

    with torch.no_grad():
        for features, labels in loader:
            features = features.to(device)
            labels = labels.to(device)

            logits = model(features)
            loss = criterion(logits, labels)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            total_loss += loss.item() * features.size(0)
            correct += (preds == labels).sum().item()
            total += features.size(0)
            true_positive += ((preds == 1) & (labels == 1)).sum().item()
            false_positive += ((preds == 1) & (labels == 0)).sum().item()
            false_negative += ((preds == 0) & (labels == 1)).sum().item()

    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    return {
        "loss": total_loss / total,
        "accuracy": correct / total,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0

    for features, labels in train_loader:
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item() * features.size(0)

    train_loss = total_train_loss / len(train_loader.dataset)
    test_metrics = evaluate(model, test_loader)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"test_loss={test_metrics['loss']:.4f} | "
        f"test_acc={test_metrics['accuracy']:.4f} | "
        f"test_f1={test_metrics['f1']:.4f}"
    )

## 7. Final metrics

In [ ]:
final_metrics = evaluate(model, test_loader)
privacy_metrics = {
    "epsilon": "N/A - no DP-SGD privacy accountant in this baseline",
    "delta": "N/A - no DP-SGD privacy accountant in this baseline",
}

reported_metrics = {**final_metrics, **privacy_metrics}
pd.Series(reported_metrics).to_frame("value")

## Current Version vs. DP-SGD Version

This notebook does **not** use Differential Privacy or DP-SGD yet. It trains a normal MLP with the Adam optimizer. The training loop calls `loss.backward()` and `optimizer.step()` directly, so gradients are computed from each batch and applied without per-sample gradient clipping, without adding Gaussian noise, and without tracking a privacy budget.

### Current metrics

- `loss`: average binary cross-entropy loss on the evaluated split. Lower is better.
- `accuracy`: fraction of all rows classified correctly.
- `precision`: among rows predicted as `>50K`, the fraction that are truly `>50K`.
- `recall`: among all true `>50K` rows, the fraction found by the model.
- `f1`: harmonic mean of precision and recall. This is useful because the Adult dataset is imbalanced, with fewer `>50K` examples than `<=50K` examples.
- `epsilon`: privacy loss value for a DP model. In this baseline it is `N/A` because no DP-SGD privacy accountant is used.
- `delta`: probability of privacy failure for a DP model. In this baseline it is `N/A` because no DP-SGD privacy accountant is used.

### Current training parameters

- `BATCH_SIZE = 256`: number of examples used per optimizer update.
- `EPOCHS = 20`: number of full passes over the training data.
- `LEARNING_RATE = 1e-3`: Adam step size.
- `hidden_dims = (128, 64, 32)`: MLP hidden layer sizes.
- `dropout = 0.2`: regularization that randomly drops hidden activations during training.
- `criterion = BCEWithLogitsLoss`: binary classification loss for one-logit output.
- `optimizer = Adam`: non-private optimizer. This is the main part that must change for DP-SGD.

### What should change when adding DP-SGD

A DP-SGD version should replace the normal optimizer step with a private training step, commonly using a library such as Opacus for PyTorch. The key changes are:

- Use per-sample gradients, because DP-SGD must clip each example's gradient before aggregation.
- Add `max_grad_norm`, also called the clipping norm. This limits how much one training example can affect the update.
- Add `noise_multiplier`, which controls how much Gaussian noise is added to the clipped gradients. More noise gives stronger privacy but usually lowers accuracy.
- Track the privacy budget `epsilon` for a chosen `delta`. Smaller `epsilon` means stronger privacy. A common starting choice for `delta` is less than `1 / len(train_dataset)`.
- Report utility metrics and privacy metrics together: `accuracy`, `precision`, `recall`, `f1`, plus `(epsilon, delta)`, `noise_multiplier`, `max_grad_norm`, sample rate, batch size, and epochs.

Expected tradeoff: after adding DP-SGD, test accuracy and F1 may decrease because gradient clipping and noise make training less exact. The experiment should compare this current non-private baseline against one or more DP-SGD settings, for example strong privacy with lower epsilon and weaker privacy with higher epsilon.

## What this notebook does

1. Loads `adult.data` for training and `adult.test` for testing.
2. Removes rows with unknown categorical values marked as `?`.
3. Converts the target into binary labels: `1` means `>50K`, `0` means `<=50K`.
4. Standardizes numeric columns so large values such as `fnlwgt` do not dominate optimization.
5. One-hot encodes categorical columns such as `workclass`, `education`, and `occupation`.
6. Trains a feed-forward MLP with hidden layers `(128, 64, 32)`.
7. Evaluates accuracy, precision, recall, and F1 on the official Adult test split.

Important modeling note: Adult contains sensitive attributes such as `sex` and `race`. The notebook currently includes them because the original dataset includes them. For fairness experiments, train a second version without those columns and compare performance and group-level metrics.